# Task 1 — training the LoS/NLoS classifier

Fine-tunes the WiFo2 baseline on `dataset/Task1/`, mirroring what
`python main.py --task_id 1` does, but with honest validation and a submission
file at the end.

**Read this before running.** The shipped data is tiny:

| file | samples | labels |
|---|---|---|
| `H_train.mat` + `L_train.mat` | 10 | yes (6x class 0, 4x class 1) |
| `H_test.mat` | 20 | **no** — `L_test.mat` does not exist |

Ten labelled samples cannot support a train/val split that means anything, and
the repo's `data_load_task_1` (`DataLoader.py:70`) crashes on the missing
`L_test.mat`. So this notebook:

1. precomputes frozen backbone features once (the backbone never trains)
2. measures performance with **stratified cross-validation**, not a held-out split
3. compares a linear probe against unfreezing the last encoder block
4. trains a final model on all 10 and writes predictions for the 20 test samples

Metric is the challenge's: **macro F1** (`train.py:78` scores `1 - f1_macro`).

Kernel: `itu-challenge`. Run from the repo root.

In [ ]:
import sys, warnings, time, json
from pathlib import Path
from types import SimpleNamespace

warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, torch, torch.nn as nn
import scipy.io as sio, h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from model import WiFo2_model
from main import freeze_wifo

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cpu')      # model is 15.7M params; cpu is fine and reproducible
print('torch', torch.__version__, '| root', ROOT)

## 1. Data

`H_train.mat` is MATLAB v7.3 (HDF5), `H_test.mat` is v7 (needs scipy). Handle both.

`LoadBatch` in `DataLoader.py` stacks real/imag into a channel dim and gives
`(B, 2, 24, 8, 128)`. Reproduced inline here so the notebook does not need the
DataLoader's `hdf5storage` path.

In [ ]:
def load_mat(path, key):
    """Read one variable from a .mat file, v7 or v7.3."""
    try:
        return sio.loadmat(path)[key]
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            a = f[key][...]
        if a.dtype.names == ('real', 'imag'):
            a = a['real'] + 1j * a['imag']
        return a.transpose(range(a.ndim)[::-1])

def to_input(H):
    """complex (B,T,A,K) -> real (B,2,T,A,K), same as DataLoader.LoadBatch"""
    return torch.stack([torch.tensor(H.real), torch.tensor(H.imag)], 1).float()

H_tr = load_mat(ROOT / 'dataset/Task1/H_train.mat', 'H')
y_np = load_mat(ROOT / 'dataset/Task1/L_train.mat', 'label').ravel().astype(int)
H_te = load_mat(ROOT / 'dataset/Task1/H_test.mat', 'H')

x_tr, x_te = to_input(H_tr).to(device), to_input(H_te).to(device)
y = torch.tensor(y_np).long().to(device)

print(f'train  {tuple(x_tr.shape)}  labels {y_np}')
print(f'test   {tuple(x_te.shape)}  labels: none (L_test.mat absent)')
print(f'class balance: {np.bincount(y_np)}')

assert not (ROOT / 'dataset/Task1/L_test.mat').exists(), 'L_test.mat now exists - use it as a real test set'

## 2. Build model, load checkpoint, freeze

Same three steps as `main.py:92-96`. `freeze_wifo(task_id=1)` leaves exactly one
layer trainable: `Fine_Tune_Layer_LoS_NLoS`, 49,154 params.

In [ ]:
def build(size='tinypro'):
    args = SimpleNamespace(
        size=size, patch_size=4, t_patch_size=4, no_qkv_bias=0,
        pos_emb='SinCos_3D', MoE=False, csi_enhance=1, rgb_pos_enhance=1,
        fastdepth_weights=str(ROOT / 'weights/FastDepthV2_L1_Best.pth'), device_id='0',
    )
    m = WiFo2_model(args=args).to(device)
    msg = m.load_state_dict(torch.load(ROOT / 'weights/model_best.pkl', map_location=device), strict=False)
    assert not [k for k in msg.missing_keys if k.startswith('blocks.')], 'encoder weights missing'
    return m, args

model, args = build()
freeze_wifo(args, model, task_id=1)

tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable {tr:,} / {sum(p.numel() for p in model.parameters()):,}')
print('trainable modules:', [n for n, p in model.named_parameters() if p.requires_grad])

## 3. Precompute frozen features

The backbone never updates, so running it every epoch is pure waste. Extract
the 384x64 latent once, flatten to 24,576 features, and train the head on
those.

`main.py` does not do this — it re-runs the full encoder for all 200 epochs.
For a linear probe the results are identical and this is far faster.

In [ ]:
@torch.no_grad()          # not inference_mode: these tensors feed autograd later
def features(m, x, batch=8):
    """Frozen encoder output, flattened. mask_ratio=0 => no tokens dropped (train.py:177)."""
    m.eval()
    out = []
    for i in range(0, len(x), batch):
        lat, *_ = m.forward_encoder(x[i:i+batch], 0.0, 'fre', scale=[1, 1, 1])
        out.append(lat.flatten(1))
    return torch.cat(out)

t0 = time.time()
F_tr = features(model, x_tr)
F_te = features(model, x_te)
print(f'train features {tuple(F_tr.shape)}   test features {tuple(F_te.shape)}')
print(f'extracted in {time.time()-t0:.1f}s  ({len(x_tr)+len(x_te)} samples)')
print(f'\n24,576 features from {len(y)} labelled samples -> {24576/len(y):.0f}x more dims than examples.')
print('A linear head can fit any labelling of 10 points exactly. Cross-validation is mandatory.')

## 4. Why a single train/val split tells you nothing

Fit the head on all 10 samples and it reaches 100% — on the data it just
memorised. That number is meaningless. Shown once so the CV results below have
something to contrast against.

In [ ]:
def new_head(in_dim=24576, out_dim=2):
    h = nn.Sequential(nn.Linear(in_dim, out_dim)).to(device)
    torch.nn.init.trunc_normal_(h[0].weight, std=0.02); torch.nn.init.zeros_(h[0].bias)
    return h

def fit_head(head, F, yy, epochs=200, lr=1e-3, clip=0.05, wd=0.0, log=None):
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.CrossEntropyLoss()
    hist = []
    for ep in range(epochs):
        opt.zero_grad()
        out = head(F)
        loss = lossf(out, yy)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), clip)
        opt.step()
        hist.append(loss.item())
        if log and ep % log == 0:
            print(f'  epoch {ep:>4}  loss {loss.item():.4f}  acc {(out.argmax(1)==yy).float().mean():.2f}')
    return hist

torch.manual_seed(SEED)
head = new_head()
hist = fit_head(head, F_tr, y, log=50)

with torch.no_grad():
    pred = head(F_tr).argmax(1)
print(f'\ntrain accuracy  {(pred==y).float().mean():.2f}')
print(f'train macro F1  {f1_score(y.cpu(), pred.cpu(), average="macro"):.2f}   <- memorised, not learned')

## 5. Honest evaluation — stratified cross-validation

5-fold stratified: each fold holds out 2 samples the head never sees, and class
balance is preserved. Every sample gets predicted exactly once by a model that
did not train on it.

With 10 samples the variance is huge. Treat the result as a sanity check on
whether the frozen features carry *any* LoS/NLoS signal, not as a performance
estimate.

In [ ]:
def cross_val(F, yy, n_splits=5, epochs=200, lr=1e-3, seed=SEED, **kw):
    Fc, yc = F.cpu().numpy(), yy.cpu().numpy()
    oof = np.zeros(len(yc), dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (itr, iva) in enumerate(skf.split(Fc, yc)):
        torch.manual_seed(seed + fold)
        h = new_head(F.shape[1])
        fit_head(h, F[itr], yy[itr], epochs=epochs, lr=lr, **kw)
        with torch.no_grad():
            oof[iva] = h(F[iva]).argmax(1).cpu().numpy()
    return oof

oof = cross_val(F_tr, y)
f1 = f1_score(y_np, oof, average='macro')

print(f"{'sample':>7}{'true':>7}{'pred':>7}{'ok':>5}")
for i in range(len(y_np)):
    print(f'{i:>7}{y_np[i]:>7}{oof[i]:>7}{"y" if oof[i]==y_np[i] else "n":>5}')

print(f'\nout-of-fold accuracy  {(oof==y_np).mean():.2f}')
print(f'out-of-fold macro F1  {f1:.3f}')
print(f'challenge score (1-F1) {1-f1:.3f}   lower is better')
print(f'\nconfusion matrix (rows=true, cols=pred):\n{confusion_matrix(y_np, oof)}')

maj = np.full_like(y_np, np.bincount(y_np).argmax())
print(f'\nmajority-class baseline macro F1: {f1_score(y_np, maj, average="macro"):.3f}')

## 6. Does anything beat the linear probe?

Three cheap variants, all evaluated with the same CV:

- **linear probe** — current baseline, 24,576 -> 2
- **mean-pooled probe** — average the 384 tokens first, then `Linear(64, 2)`. 130 params instead of 49,154, and it survives a change of input size (the flattened head hardcodes 384)
- **weight decay** — regularise the 24,576-input probe

If mean-pooling wins, the baseline head is overparameterised for this data.

In [ ]:
F_pool = F_tr.reshape(len(F_tr), 384, -1).mean(1)          # (10, 64)
print('pooled features', tuple(F_pool.shape))

variants = {
    'linear probe (24576->2)'  : (F_tr,   dict()),
    'linear probe + wd=1e-2'   : (F_tr,   dict(wd=1e-2)),
    'mean-pool probe (64->2)'  : (F_pool, dict(lr=1e-2)),
    'mean-pool + wd=1e-2'      : (F_pool, dict(lr=1e-2, wd=1e-2)),
}

print(f"\n{'variant':28s}{'params':>10s}{'acc':>8s}{'macro F1':>11s}{'1-F1':>8s}")
results = {}
for name, (F, kw) in variants.items():
    o = cross_val(F, y, **kw)
    s = f1_score(y_np, o, average='macro')
    results[name] = s
    print(f'{name:28s}{F.shape[1]*2+2:>10,}{(o==y_np).mean():>8.2f}{s:>11.3f}{1-s:>8.3f}')

best = max(results, key=results.get)
print(f'\nbest: {best}  (macro F1 {results[best]:.3f})')
print('With 10 samples these differences are within noise. Rerun with a different SEED to see the spread.')

## 7. Unfreezing the last encoder block

A linear probe can only work if the frozen features are already separable.
Unfreezing `blocks[5]` adds 25,216 trainable params and lets the backbone adapt
its final representation.

This cannot use cached features — the encoder now changes every step — so it is
slower. Still seconds at `tinypro` scale.

In [ ]:
def fit_full(x, yy, epochs=60, lr=1e-4, unfreeze_last=True, seed=SEED):
    torch.manual_seed(seed)
    m, a = build()
    freeze_wifo(a, m, task_id=1)
    torch.nn.init.trunc_normal_(m.Fine_Tune_Layer_LoS_NLoS[0].weight, std=0.02)
    torch.nn.init.zeros_(m.Fine_Tune_Layer_LoS_NLoS[0].bias)
    if unfreeze_last:
        for p in m.blocks[-1].parameters():
            p.requires_grad = True
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    lossf = nn.CrossEntropyLoss()
    m.train()
    for ep in range(epochs):
        opt.zero_grad()
        out = m(x, mask_ratio=0.0, mask_strategy='fre', task_id=1)
        loss = lossf(out, yy)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 0.05)
        opt.step()
    m.eval()
    return m, sum(p.numel() for p in params)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_ft = np.zeros(len(y_np), dtype=int)
t0 = time.time()
for fold, (itr, iva) in enumerate(skf.split(F_tr.cpu().numpy(), y_np)):
    m, np_ = fit_full(x_tr[itr], y[itr], seed=SEED + fold)
    with torch.inference_mode():
        oof_ft[iva] = m(x_tr[iva], mask_ratio=0.0, mask_strategy='fre', task_id=1).argmax(1).cpu().numpy()
    print(f'fold {fold}  done')

f1_ft = f1_score(y_np, oof_ft, average='macro')
print(f'\ntrainable params      {np_:,}  (head 49,154 + last block 25,216)')
print(f'out-of-fold accuracy  {(oof_ft==y_np).mean():.2f}')
print(f'out-of-fold macro F1  {f1_ft:.3f}   vs linear probe {results["linear probe (24576->2)"]:.3f}')
print(f'took {time.time()-t0:.1f}s')

## 8. Train the final model on all 10 samples

Once you have picked a recipe from the CV above, retrain on everything — no
sense discarding 20% of ten samples. This uses the repo's own head so the
saved checkpoint stays loadable by `main.py`.

In [ ]:
torch.manual_seed(SEED)
final, args = build()
freeze_wifo(args, final, task_id=1)
head = final.Fine_Tune_Layer_LoS_NLoS
torch.nn.init.trunc_normal_(head[0].weight, std=0.02); torch.nn.init.zeros_(head[0].bias)

F_all = features(final, x_tr)
hist = fit_head(head, F_all, y, epochs=200, lr=1e-3, log=50)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(hist); ax.set(xlabel='epoch', ylabel='cross-entropy', title='final head training loss', yscale='log')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 9. Predict the 20 unlabelled test samples

`H_test.mat` is the challenge submission set. No labels ship with it, so there
is nothing to score locally — check the class distribution for sanity instead.
A model predicting one class for all 20 is broken regardless of its CV score.

In [ ]:
with torch.inference_mode():
    logits = final(x_te, mask_ratio=0.0, mask_strategy='fre', task_id=1)
    prob = logits.softmax(-1)
pred_te = prob.argmax(-1).cpu().numpy()

print(f"{'sample':>7}{'pred':>7}{'P(0)':>9}{'P(1)':>9}{'confidence':>12}")
for i in range(len(pred_te)):
    print(f'{i:>7}{pred_te[i]:>7}{prob[i,0]:>9.3f}{prob[i,1]:>9.3f}{prob[i].max():>12.3f}')

print(f'\npredicted distribution {np.bincount(pred_te, minlength=2)}  (train was {np.bincount(y_np)})')
print(f'mean confidence {prob.max(1).values.mean():.3f}')
if len(np.unique(pred_te)) == 1:
    print('WARNING: all test samples got the same class. Model is degenerate.')

## 10. Save checkpoint and submission

The grader does `json.load(f)` on **one** file holding all three tasks:

```json
{"task1": [1, 0, 0, ...], "task2": [[[[...]]]], "task3": [...]}
```

`task1` is a flat list of 20 ints, one per test sample, in file order. A CSV
here fails with `JSONDecodeError: Expecting value: line 1 column 1 (char 0)` —
the grader parses before it validates anything.

The grader reads all three keys unconditionally
(`score_task2 = evaluate_task2(..., np.array(submission["task2"]).flatten())`),
so a `task1`-only file dies with `KeyError: 'task2'`. This notebook therefore
always emits all three keys, filling Tasks 2 and 3 with empty lists unless
`PREV_SUBMISSION` points at a file that already has real ones.

Empty lists get past the `KeyError`, but `evaluate_task2` then compares a
length-0 array against ground truth and will most likely raise or return
garbage — this is a way to get Task 1 graded, not a scoring submission for the
other two.

Two things that silently break the file:

- `np.int64` is not JSON-serialisable — cast with `int()`, which `.tolist()` does
- length must be exactly 20; a shorter list means the test set was misread

The checkpoint keeps the repo's `state_dict` layout, so `main.py --task Test`
can load it with `--wifo_weights`. `res18` and `fast_depth` are dropped before
saving: they belong to Tasks 2 and 3, `__init__` reloads them from their own
sources every time, and keeping them makes the file 63 MB instead of 1.9 MB.

In [ ]:
out = ROOT / 'experiments/notebook_task1'
out.mkdir(parents=True, exist_ok=True)

# point this at your last accepted submission to keep its task2/task3 entries
PREV_SUBMISSION = None          # e.g. ROOT / 'experiments/submission.json'

sd = {k: v for k, v in final.state_dict().items() if not k.startswith(('res18.', 'fast_depth.'))}
torch.save(sd, out / 'model_best.pkl')

submission = {}
if PREV_SUBMISSION and Path(PREV_SUBMISSION).exists():
    submission = json.loads(Path(PREV_SUBMISSION).read_text())
    print(f'merging into {PREV_SUBMISSION} (keys: {sorted(submission)})')

submission['task1'] = pred_te.tolist()          # .tolist() casts np.int64 -> int
submission.setdefault('task2', [])              # blank placeholder - grader needs the key present
submission.setdefault('task3', [])

assert len(submission['task1']) == len(x_te) == 20, f"task1 must hold 20 labels, got {len(submission['task1'])}"
assert all(isinstance(v, int) for v in submission['task1']), 'labels must be plain ints'

sub_path = out / 'submission.json'
sub_path.write_text(json.dumps(submission))

# parse it back exactly as the grader does, before uploading
reloaded = json.loads(sub_path.read_text())
print(f"task1 -> {reloaded['task1']}")
print(f"keys in submission: {sorted(reloaded)}")
for t in ('task2', 'task3'):
    if not reloaded[t]:
        print(f'NOTE: {t} is blank. Key present so the grader parses, but {t} will not score.')

(out / 'cv_results.json').write_text(json.dumps({
    'oof_macro_f1': float(f1),
    'oof_accuracy': float((oof == y_np).mean()),
    'variants': {k: float(v) for k, v in results.items()},
    'finetune_last_block_macro_f1': float(f1_ft),
    'n_train': int(len(y_np)), 'seed': SEED,
}, indent=2))

print()
for p in sorted(out.iterdir()):
    print(f'{p.relative_to(ROOT)}  {p.stat().st_size/1e3:.1f} KB')

## 11. The repo's own training path

Equivalent CLI run, once `L_test.mat` exists:

```bash
python main.py --task Train --task_id 1 --size tinypro \
    --wifo_weights ./weights/model_best.pkl \
    --lr 1e-5 --epochs 200 --early_stop 5 --batch_size 64 --clip_grad 0.05
```

Differences from this notebook:

| | `main.py` | this notebook |
|---|---|---|
| validation | held-out `H_test`/`L_test` | 5-fold stratified CV on the 10 train samples |
| features | encoder re-run every epoch | cached once (backbone is frozen) |
| lr | `1e-5` | `1e-3` — `1e-5` barely moves a from-scratch head in 200 epochs |
| test labels | required | not required |

`data_load_task_1` (`DataLoader.py:70`) will raise on the missing `L_test.mat`
until the full challenge dataset is in place.

## What to fix when the real dataset arrives

- drop the CV in favour of the provided split once there are enough samples
- revisit `lr`; `1e-5` is sane for a warm-started head, not a random one
- try `mask_ratio > 0` during training as token dropout — free regularisation, `mask_strategy='fre'` already supports it
- replace the flattened head with mean-pooling if section 6 favours it; it drops 49,024 params and unbinds the model from a 384-token input